# Week 4: LLM Benchmarking vs Week 3 ML

Cells in this notebook were taken from the tutorial written by Victoria Reynolds.

Steps:
- Load Week 3 test data
- Query LLM endpoint
- Parse outputs into structured predictions
- Compare ML vs each LLM
- Save raw, clean, and summary result CSV files

## Step 1: Imports

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from datetime import datetime

openai = __import__('openai')
OpenAI = openai.OpenAI

In [20]:
# =============================================================================
# LLM Pressure Threshold Classification — Sampled Evaluation
# 100 samples per split period (Train/Test/Validate) = 300 total
# =============================================================================
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
from pathlib import Path
from openai import OpenAI
import json

# =============================================================================
# CONFIGURATION
# =============================================================================
ROOT             = Path('..').resolve()
CLEANED_CSV      = 'week3_artifacts/FTES_1hour_cleaned.csv'
# OUTPUT_DIR       = ROOT / 'Results'
# VIZ_DIR          = ROOT / 'Visualizations' / 'LLM'
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# VIZ_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL           = 'Time'
PRESSURE_COL       = 'Injection Pressure'
INJECTION_FLOW     = 'Net Flow'
PRODUCER_FLOW      = 'TN Interval Flow'
PRESSURE_THRESHOLD = 5000.0
PRESSURE_ALERT     = 4800.0
HORIZONS           = [1, 6, 12, 24]
SAMPLES_PER_SPLIT  = 100
ITERATION_NUMBER   = 2   # increment to avoid overwriting v1 results

TRAIN_END = "2025-01-20 00:00:00"
TEST_END  = "2025-02-01 00:00:00"

API_KEY = 'EMPTY'
MODEL_ENDPOINTS = [
    {
        'label'     : 'gemma-3-12b-it',
        'model_name': 'gemma-3-12b-it',
        'host'      : 'localhost',
        'port'      : 8002,
    },
]
endpoint_cfg = MODEL_ENDPOINTS[0]

# =============================================================================
# SYSTEM PROMPT — updated with stability/regime guidance
# =============================================================================
SYSTEM_PROMPT = """You are a careful Fracture Thermal Energy Storage \
operational safety monitor.
Your task is to assess whether Injection Pressure will exceed the \
operational threshold of 5000 psi at multiple future horizons, \
using only features measured at time t.

Physical context:
- headroom = 5000 - current pressure (negative = already exceeded)
- hours_above_threshold: consecutive hours pressure has been above 5000 psi.
  If > 0 threshold is already exceeded. High values = sustained exceedance.
- P_roll_std_24h: std of pressure over last 24h.
  LOW (< 5 psi) = pressure is STABLE. HIGH = actively changing.
- regime: current operational state —
    above_stabilising : above threshold, pressure flat or falling
    above_rising      : above threshold AND still rising
    alert_rising      : 4800-5000 psi band, rising → crossing likely soon
    alert_stable      : 4800-5000 psi band, stable/falling
    below_rising_fast : below alert but rising quickly
    below_stable      : well below threshold, stable
- Q_reduction: negative = operator cutting injection → pressure likely to fall
- d2P_dt2: pressure acceleration. Positive = rate of rise increasing.

Key decision rules:
- hours_above_threshold > 0 AND P_roll_std_24h < 10:
    pressure STABLE above threshold → above at short horizons,
    reassess long horizons if Q_reduction is active
- regime = above_stabilising AND Q_reduction < -0.1:
    operator reducing pressure → long horizon may return below
- regime = alert_rising AND dP_dt_6h > 1:
    threshold crossing imminent → above likely at +1h and +6h
- regime = below_stable AND headroom > 500:
    safe → below at all horizons with high confidence

Confidence rules:
- Strong sustained trend (dP_dt_6h consistent): higher confidence
- headroom < 100: reduce confidence, outcome uncertain
- P_roll_std_24h < 5 (very stable): increase confidence in current state
- Q_reduction active: reduce confidence at longer horizons

No markdown, no prose, no code fences, no extra keys.
Required JSON schema:
{
  "t+1h":  {"classification": "above or below", "confidence": <0_to_1_float>},
  "t+6h":  {"classification": "above or below", "confidence": <0_to_1_float>},
  "t+12h": {"classification": "above or below", "confidence": <0_to_1_float>},
  "t+24h": {"classification": "above or below", "confidence": <0_to_1_float>}
}
"""

# =============================================================================
# FEATURES
# =============================================================================
FEAT_COLS = [
    'Injection Pressure',
    'headroom',
    'in_alert_band',
    'dP_dt_1h',
    'dP_dt_6h',
    'd2P_dt2',
    'P_roll_std_6h',
    'P_roll_std_24h',
    'hours_above_threshold',
    'Net Flow',
    'Q_reduction',
    'Q_roll_std_6h',
    'net_subsurface',
    'cumulative_net',
    'regime',
]


def _hours_since_active(series: pd.Series, threshold: float) -> pd.Series:
    values = series.values
    result = np.zeros(len(values))
    count  = 0
    for i, v in enumerate(values):
        count = 0 if v < threshold else count + 1
        result[i] = count
    return pd.Series(result, index=series.index)


def assign_regime(row) -> str:
    if row['headroom'] < 0 and row['dP_dt_6h'] <= 0:
        return 'above_stabilising'
    elif row['headroom'] < 0 and row['dP_dt_6h'] > 0:
        return 'above_rising'
    elif row['in_alert_band'] and row['dP_dt_6h'] > 0:
        return 'alert_rising'
    elif row['in_alert_band'] and row['dP_dt_6h'] <= 0:
        return 'alert_stable'
    elif row['dP_dt_6h'] > 2:
        return 'below_rising_fast'
    else:
        return 'below_stable'


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out['net_subsurface']        = out[INJECTION_FLOW] - out[PRODUCER_FLOW]
    out['cumulative_net']        = out['net_subsurface'].cumsum()
    out['dP_dt_1h']              = out[PRESSURE_COL].diff(1).fillna(0)
    out['dP_dt_6h']              = (out[PRESSURE_COL].diff(6) / 6).fillna(0)
    out['d2P_dt2']               = out['dP_dt_1h'].diff().fillna(0)
    out['headroom']              = PRESSURE_THRESHOLD - out[PRESSURE_COL]
    out['in_alert_band']         = (
        (out[PRESSURE_COL] >= PRESSURE_ALERT) &
        (out[PRESSURE_COL] <  PRESSURE_THRESHOLD)
    ).astype(int)
    out['P_roll_std_6h']         = out[PRESSURE_COL].rolling(6).std().fillna(0)
    out['P_roll_std_24h']        = out[PRESSURE_COL].rolling(24).std().fillna(0)
    out['hours_above_threshold'] = _hours_since_active(
        out[PRESSURE_COL], PRESSURE_THRESHOLD)
    out['Q_reduction']           = out[INJECTION_FLOW].diff().clip(upper=0).fillna(0)
    out['Q_roll_std_6h']         = out[INJECTION_FLOW].rolling(6).std().fillna(0)
    out['regime']                = out.apply(assign_regime, axis=1)

    # Ground truth labels
    for h in HORIZONS:
        out[f'actual_above_{h}h'] = (
            out[PRESSURE_COL]
            .shift(-h)
            .apply(lambda p: 1 if not pd.isna(p)
                   and p >= PRESSURE_THRESHOLD else 0)
        )
    return out


# =============================================================================
# LOAD & SPLIT
# =============================================================================
def load_and_split(csv_path: str):
    raw = pd.read_csv(csv_path, parse_dates=[TIME_COL])
    raw = raw.sort_values(TIME_COL).reset_index(drop=True)

    missing = {PRESSURE_COL, INJECTION_FLOW, PRODUCER_FLOW} - set(raw.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')

    df = raw[[TIME_COL, PRESSURE_COL, INJECTION_FLOW, PRODUCER_FLOW]].copy()
    df = prepare_features(df)

    # Drop warmup and tail NaNs
    feat_cols  = [c for c in FEAT_COLS if c != 'regime']
    label_cols = [f'actual_above_{h}h' for h in HORIZONS]
    df = df.dropna(subset=feat_cols + label_cols).reset_index(drop=True)

    train_df    = df[df[TIME_COL] <= TRAIN_END].copy()
    test_df     = df[(df[TIME_COL] > TRAIN_END) &
                     (df[TIME_COL] <= TEST_END)].copy()
    validate_df = df[df[TIME_COL] > TEST_END].copy()

    print(f'Loaded : {csv_path}')
    print(f'  Train    : {len(train_df)} rows')
    print(f'  Test     : {len(test_df)} rows')
    print(f'  Validate : {len(validate_df)} rows')
    return train_df, test_df, validate_df


# =============================================================================
# STRATIFIED SAMPLE
# Ensures sample covers both above and below threshold rows
# proportional to their occurrence in each split
# =============================================================================
def stratified_sample(df:         pd.DataFrame,
                       n:          int,
                       label_col:  str = 'actual_above_1h',
                       seed:       int = 42) -> pd.DataFrame:
    """
    Sample n rows stratified by above/below threshold.
    Preserves class balance from the original split.
    Uses +1h label for stratification — consistent across horizons
    since the threshold period is a sustained block.
    """
    above = df[df[label_col] == 1]
    below = df[df[label_col] == 0]

    # Proportion of above/below in original split
    frac_above = len(above) / len(df)
    n_above    = min(int(round(n * frac_above)), len(above))
    n_below    = min(n - n_above, len(below))

    sampled = pd.concat([
        above.sample(n=n_above, random_state=seed),
        below.sample(n=n_below, random_state=seed),
    ]).sort_values(TIME_COL).reset_index(drop=True)

    print(f'  Sampled {len(sampled)} rows  '
          f'({n_above} above / {n_below} below)')
    return sampled


# =============================================================================
# FEW-SHOT — from train only
# =============================================================================
def build_few_shot_examples(train_df:  pd.DataFrame,
                             feat_cols: list[str],
                             horizons:  list[int]) -> list[dict]:
    max_h    = max(horizons)
    valid    = train_df.iloc[:-(max_h + 1)]
    examples = []

    scenarios = [
        ('rising_below_alert',
         (valid[PRESSURE_COL] < PRESSURE_ALERT) & (valid['dP_dt_6h'] > 0)),
        ('in_alert_band',
         (valid[PRESSURE_COL] >= PRESSURE_ALERT) &
         (valid[PRESSURE_COL] <  PRESSURE_THRESHOLD) &
         (valid['dP_dt_1h'] > 0)),
        ('above_threshold',
         valid[PRESSURE_COL] >= PRESSURE_THRESHOLD),
    ]

    for name, condition in scenarios:
        candidates = valid[condition]
        if len(candidates) == 0:
            continue
        row = candidates.iloc[0]
        loc = train_df.index.get_loc(row.name)

        output = {}
        for h in horizons:
            if loc + h >= len(train_df):
                continue
            future_p = float(train_df.iloc[loc + h][PRESSURE_COL])
            label    = 'above' if future_p >= PRESSURE_THRESHOLD else 'below'
            headroom_now = float(row['headroom'])
            dp_6h        = float(row['dP_dt_6h'])

            if label == 'above' and headroom_now < 0:
                conf = round(max(0.60, 0.95 - h * 0.01), 2)
            elif label == 'above' and headroom_now < 200 and dp_6h > 0:
                conf = round(max(0.55, 0.88 - h * 0.015), 2)
            elif label == 'below' and headroom_now > 500:
                conf = round(max(0.55, 0.90 - h * 0.012), 2)
            else:
                conf = round(max(0.50, 0.75 - h * 0.015), 2)

            output[f't+{h}h'] = {'classification': label, 'confidence': conf}

        if output:
            examples.append({
                't'             : pd.Timestamp(row[TIME_COL]).strftime(
                    '%-m/%-d/%Y %I:%M:%S %p'),
                'scenario'      : name,
                'input_features': {
                    col: round(float(row[col]), 4)
                    if col != 'regime' else str(row[col])
                    for col in feat_cols
                    if col in train_df.columns
                },
                'output_json': output,
            })

        if len(examples) >= 3:
            break

    print(f'Few-shot examples: {len(examples)} (from train period)')
    return examples


# =============================================================================
# PROMPT BUILDER
# =============================================================================
def build_prompt(time_t:        str,
                 feature_names: list[str],
                 features_t:    dict,
                 horizons:      list[int],
                 few_shot:      list[dict] | None = None) -> str:

    headroom  = features_t.get('headroom', 9999)
    dp_1h     = features_t.get('dP_dt_1h', 0)
    dp_6h     = features_t.get('dP_dt_6h', 0)
    current_p = features_t.get('Injection Pressure', 0)
    regime    = features_t.get('regime', 'unknown')
    h_above   = features_t.get('hours_above_threshold', 0)
    std_24h   = features_t.get('P_roll_std_24h', 0)

    if current_p >= PRESSURE_THRESHOLD and std_24h < 10:
        status = (f"STABLE ABOVE THRESHOLD — {abs(headroom):.1f} psi over limit, "
                  f"stable for ~{h_above:.0f}h (std={std_24h:.1f} psi)")
    elif current_p >= PRESSURE_THRESHOLD:
        status = (f"ABOVE THRESHOLD — {abs(headroom):.1f} psi over limit, "
                  f"actively {'rising' if dp_1h > 0 else 'falling'}")
    elif features_t.get('in_alert_band'):
        status = (f"IN ALERT BAND — {headroom:.1f} psi below threshold, "
                  f"pressure {'rising' if dp_1h > 0 else 'falling'}")
    elif headroom < 500 and dp_6h > 0:
        status = f"APPROACHING — {headroom:.1f} psi headroom, sustained rise"
    else:
        status = (f"NORMAL — {headroom:.1f} psi headroom, "
                  f"{'rising' if dp_1h > 0 else 'stable/falling'}")

    horizons_str = ', '.join([f't+{h}h' for h in horizons])

    prompt_lines = [
        f'Task: Classify whether Injection Pressure will exceed '
        f'5000 psi at horizons: {horizons_str}',
        f'Current timestamp : {time_t}',
        f'Operational status: {status}',
        f'Regime            : {regime}',
        '',
        'Key signals:',
        f'  Current pressure       : {current_p:.2f} psi',
        f'  Headroom to 5000 psi   : {headroom:.2f} psi',
        f'  Hours above threshold  : {h_above:.0f}h',
        f'  Pressure stability 24h : {std_24h:.2f} psi std',
        f'  dP/dt last 1h          : {dp_1h:+.2f} psi/hr',
        f'  dP/dt last 6h          : {dp_6h:+.2f} psi/hr',
        '',
        'Full feature snapshot (use only these):',
        json.dumps(features_t, indent=2),
        '',
        'Classify "above" if pressure will exceed 5000 psi, '
        '"below" if it will not.',
        'Return JSON only. No explanation.',
        'Required output:',
        json.dumps(
            {f't+{h}h': {'classification': 'above or below',
                         'confidence': '0_to_1_float'}
             for h in horizons},
            indent=2
        ),
    ]

    if few_shot:
        prompt_lines += ['', '=' * 40, 'Examples:', '=' * 40]
        for i, ex in enumerate(few_shot, start=1):
            prompt_lines += [
                f'\nExample {i} ({ex.get("scenario", "")}) t={ex["t"]}:',
                json.dumps(ex['input_features'], indent=2),
                'Correct output:',
                json.dumps(ex['output_json'], indent=2),
            ]
        prompt_lines.append('=' * 40)

    prompt_lines += ['', 'Now classify:']
    return '\n'.join(prompt_lines)


# =============================================================================
# LLM
# =============================================================================
def query_llm(prompt: str, endpoint_cfg: dict,
              temperature: float = 0.0, seed: int = 0) -> str:
    client = OpenAI(
        api_key  = API_KEY,
        base_url = f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1",
    )
    response = client.chat.completions.create(
        model       = endpoint_cfg['model_name'],
        messages    = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ],
        temperature = temperature,
        seed        = seed,
    )
    return response.choices[0].message.content


def parse_response(response_text: str,
                   horizons: list[int] = HORIZONS) -> dict | None:
    try:
        text = response_text.strip()
        if '```' in text:
            text = text.split('```')[1]
            if text.startswith('json'):
                text = text[4:]
        text   = text.strip()
        parsed = json.loads(text)

        result = {}
        for h in horizons:
            key = f't+{h}h'
            if key not in parsed:
                print(f'  ⚠ Missing key {key}')
                return None
            clf  = parsed[key].get('classification', '').lower().strip()
            conf = float(parsed[key].get('confidence', 0.5))
            if clf not in ('above', 'below'):
                print(f'  ⚠ Invalid classification: {clf}')
                return None
            result[key] = {
                'classification': clf,
                'confidence'    : round(conf, 4),
                'binary'        : 1 if clf == 'above' else 0,
            }
        return result
    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f'  ⚠ Parse error: {e}')
        return None


# =============================================================================
# EVALUATE
# =============================================================================
def compute_metrics(df: pd.DataFrame,
                    horizons: list[int]) -> pd.DataFrame:
    rows = []
    for h in horizons:
        pred_col = f't+{h}h_binary'
        true_col = f'actual_above_{h}h'
        valid    = df[[pred_col, true_col]].dropna()
        if len(valid) == 0:
            continue

        y_pred = valid[pred_col].astype(int)
        y_true = valid[true_col].astype(int)

        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
        recall    = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        f1        = (2 * precision * recall / (precision + recall)
                     if not (np.isnan(precision) or np.isnan(recall))
                     and (precision + recall) > 0 else float('nan'))

        rows.append({
            'Horizon'  : f'+{h}h',
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'Precision': round(precision, 3) if not np.isnan(precision) else '—',
            'Recall'   : round(recall, 3)    if not np.isnan(recall)    else '—',
            'F1'       : round(f1, 3)        if not np.isnan(f1)        else '—',
            'n'        : len(valid),
        })
    return pd.DataFrame(rows)


# =============================================================================
# CONFUSION MATRIX PLOT
# =============================================================================
def plot_confusion_matrices(results_by_split: dict[str, pd.DataFrame],
                             horizons:         list[int]):
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    splits  = list(results_by_split.keys())
    n_s     = len(splits)
    n_h     = len(horizons)
    fig, axes = plt.subplots(n_s, n_h,
                             figsize=(3.5 * n_h, 3.5 * n_s))
    fig.suptitle(f'LLM Threshold Classification\n'
                 f'Threshold = {PRESSURE_THRESHOLD:.0f} psi  '
                 f'({SAMPLES_PER_SPLIT} samples/split)',
                 fontsize=11, y=1.02)

    for row, split_name in enumerate(splits):
        df = results_by_split[split_name]
        for col, h in enumerate(horizons):
            ax       = axes[row, col] if n_s > 1 else axes[col]
            pred_col = f't+{h}h_binary'
            true_col = f'actual_above_{h}h'
            valid    = df[[pred_col, true_col]].dropna()

            y_pred = valid[pred_col].astype(int)
            y_true = valid[true_col].astype(int)

            cm   = confusion_matrix(y_true, y_pred, labels=[0, 1])
            disp = ConfusionMatrixDisplay(cm, display_labels=['Below', 'Above'])
            disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')

            tp = cm[1, 1]; fp = cm[0, 1]
            fn = cm[1, 0]; tn = cm[0, 0]
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1        = (2 * precision * recall / (precision + recall)
                         if (precision + recall) > 0 else 0.0)

            ax.text(0.98, 0.02,
                    f'P={precision:.3f}\nR={recall:.3f}\nF1={f1:.3f}',
                    transform=ax.transAxes, fontsize=7,
                    va='bottom', ha='right',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='lightyellow', alpha=0.85))

            ax.set_title(f'+{h}h' if row == 0 else '', fontsize=9)
            ax.set_ylabel(f'{split_name}\nActual' if col == 0 else '',
                          fontsize=8)
            ax.set_xlabel('Predicted' if row == n_s - 1 else '', fontsize=8)

    plt.tight_layout()
    out_path = VIZ_DIR / f'llm_confusion_matrices_v{ITERATION_NUMBER}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_path}')



# ── load & split ──────────────────────────────────────────────────────────
train_df, test_df, validate_df = load_and_split(CLEANED_CSV)

# ── few-shot from train only ───────────────────────────────────────────────
few_shot = build_few_shot_examples(train_df, FEAT_COLS, HORIZONS)

# ── stratified sample from each split ─────────────────────────────────────
print(f'\nSampling {SAMPLES_PER_SPLIT} rows per split:')
splits = {
    'Train'   : stratified_sample(train_df,    SAMPLES_PER_SPLIT),
    'Test'    : stratified_sample(test_df,     SAMPLES_PER_SPLIT),
    'Validate': stratified_sample(validate_df, SAMPLES_PER_SPLIT),
}
total = sum(len(v) for v in splits.values())
print(f'Total rows to evaluate: {total}')

# ── run LLM ───────────────────────────────────────────────────────────────
results_by_split = {}

for split_name, sample_df in splits.items():
    print(f'\nRunning {split_name} ({len(sample_df)} rows)...')
    results  = []
    n_failed = 0

    for i, row in sample_df.iterrows():
        time_t     = pd.Timestamp(row[TIME_COL]).strftime(
            '%-m/%-d/%Y %I:%M:%S %p')
        features_t = {
            col: (round(float(row[col]), 4)
                  if col != 'regime' else str(row[col]))
            for col in FEAT_COLS
            if col in sample_df.columns
        }

        prompt = build_prompt(
            time_t        = time_t,
            feature_names = FEAT_COLS,
            features_t    = features_t,
            horizons      = HORIZONS,
            few_shot      = few_shot,
        )

        raw    = query_llm(prompt, endpoint_cfg,
                           temperature=0.0, seed=0)
        parsed = parse_response(raw, HORIZONS)

        if parsed is None:
            n_failed += 1
            continue

        result_row = {TIME_COL: row[TIME_COL]}
        for h in HORIZONS:
            key = f't+{h}h'
            result_row[f'{key}_classification'] = parsed[key]['classification']
            result_row[f'{key}_confidence']     = parsed[key]['confidence']
            result_row[f'{key}_binary']         = parsed[key]['binary']
            result_row[f'actual_above_{h}h']    = row[f'actual_above_{h}h']
        results.append(result_row)

        if (i + 1) % 25 == 0:
            print(f'  {i+1}/{len(sample_df)}  ({n_failed} failed)')

    split_results = pd.DataFrame(results)
    split_results['split'] = split_name
    results_by_split[split_name] = split_results
    print(f'  Done: {len(split_results)} rows, {n_failed} failed')

# ── save ──────────────────────────────────────────────────────────────────
all_results = pd.concat(results_by_split.values(), ignore_index=True)
out_path    = f'llm_classifications.csv'
all_results.to_csv(out_path, index=False)
print(f'\nResults saved → {out_path}')

# ── metrics ───────────────────────────────────────────────────────────────
print(f'\n{"="*55}')
for split_name, split_df in results_by_split.items():
    metrics = compute_metrics(split_df, HORIZONS)
    print(f'\n  LLM — {split_name}')
    print(f'{"="*55}')
    print(metrics.to_string(index=False))

print(f'\n{"="*55}')
print(f'  KF BASELINE (Validate) — reference')
print(f'{"="*55}')
print('  +1h  : P=0.990 R=1.000 F1=0.995')
print('  +6h  : P=0.959 R=1.000 F1=0.979')
print('  +12h : P=0.938 R=1.000 F1=0.968')
print('  +24h : P=0.872 R=0.851 F1=0.861')

# ── plot ──────────────────────────────────────────────────────────────────
# plot_confusion_matrices(results_by_split, HORIZONS)



Loaded : week3_artifacts/FTES_1hour_cleaned.csv
  Train    : 893 rows
  Test     : 288 rows
  Validate : 521 rows
Few-shot examples: 3 (from train period)

Sampling 100 rows per split:
  Sampled 100 rows  (7 above / 93 below)
  Sampled 100 rows  (92 above / 8 below)
  Sampled 100 rows  (37 above / 63 below)
Total rows to evaluate: 300

Running Train (100 rows)...
  25/100  (0 failed)
  50/100  (0 failed)
  75/100  (0 failed)
  100/100  (0 failed)
  Done: 100 rows, 0 failed

Running Test (100 rows)...
  25/100  (0 failed)
  50/100  (0 failed)
  75/100  (0 failed)
  100/100  (0 failed)
  Done: 100 rows, 0 failed

Running Validate (100 rows)...
  25/100  (0 failed)
  50/100  (0 failed)
  75/100  (0 failed)
  100/100  (0 failed)
  Done: 100 rows, 0 failed

Results saved → llm_classifications_v3.csv


  LLM — Train
Horizon  TP  FP  FN  TN  Precision  Recall    F1   n
    +1h   7  38   0  55      0.156     1.0 0.269 100
    +6h   9  36   0  55      0.200     1.0 0.333 100
   +12h   9  36   0

## Step 2: Paths and settings

In [2]:
ROOT = Path('..').resolve()

WEEK3_TEST_CSV = 'week3_artifacts/FTES_1hour_test.csv'
WEEK3_ML_RESULTS_CSV = 'week3_artifacts/xgb_test_predictions.csv'
WEEK3_ML_CONFIG_JSON = 'week3_artifacts/xgb_training_config.json'

with open(WEEK3_ML_CONFIG_JSON, "r", encoding="utf-8") as f:
    train_cfg = json.load(f)

TIME_COL = 'Time'
FEAT_COLS = train_cfg.get('feature_cols')
TRUE_VAL_COL = 'Injection Pressure'
ML_PRED_COL = 'Injection Pressure__pred'

# Only using gemma-3-12b-it
LLM_MODEL = "gemma3"
MODEL_ENDPOINTS = [  
    # {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
    # {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it', 'model_name': 'gemma-3-12b-it', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

PROMPT_DIR = ROOT / 'Prompts'
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = ROOT / 'Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_RESULTS_PATH = OUTPUT_DIR / f'{LLM_MODEL}_results_raw_{ITERATION_NUMBER}.csv'
CLEAN_RESULTS_PATH = OUTPUT_DIR / f'{LLM_MODEL}_results_clean_{ITERATION_NUMBER}.csv'
SUMMARY_RESULTS_PATH = OUTPUT_DIR / f'{LLM_MODEL}_results_summary_{ITERATION_NUMBER}.csv'

FileNotFoundError: [Errno 2] No such file or directory: 'week3_artifacts/xgb_training_config.json'

## Step 3: Load Week 3 files

In [7]:
test_df = pd.read_csv(WEEK3_TEST_CSV)
ml_df = pd.read_csv(WEEK3_ML_RESULTS_CSV)

needed_test = {TIME_COL, *FEAT_COLS, TRUE_VAL_COL}
needed_ml = {TIME_COL, ML_PRED_COL}
if needed_test - set(test_df.columns):
    raise ValueError(f'Missing in {WEEK3_TEST_CSV}: {sorted(needed_test - set(test_df.columns))}')
if needed_ml - set(ml_df.columns):
    raise ValueError(f'Missing in {WEEK3_ML_RESULTS_CSV}: {sorted(needed_ml - set(ml_df.columns))}')

df = test_df[[TIME_COL, *FEAT_COLS, TRUE_VAL_COL]].merge(
    ml_df[[TIME_COL, ML_PRED_COL]], on=TIME_COL, how='left'
)

print(f'Loaded rows in test set: {len(df)}')
# display(df.head())

Loaded rows in test set: 256


## Step 4: Prompt + parser helpers

What this does: defines a strict prompt format and a parser that turns model text into structured fields (prediction, confidence, parse status).

Why it matters: LLM responses are free-form by default, but evaluation requires deterministic, machine-readable outputs.

Principle: constrain generation and validate parsing. Reliability comes from clear output contracts plus defensive checks.

In [5]:
# =============================================================================
# CONFIGURATION
# =============================================================================
HORIZONS           = [1, 6, 12, 24]
PRESSURE_THRESHOLD = 5000.0
PRESSURE_ALERT     = 4800.0

SYSTEM_PROMPT = """You are a careful Fracture Thermal Energy Storage \
operational safety monitor.
Your task is to assess whether Injection Pressure will exceed the \
operational threshold of 5000 psi at multiple future horizons, \
using only features measured at time t.

Do not use future information, external knowledge, or assumptions \
not present in the input.
Return exactly one JSON object and nothing else.

Physical context:
- Injection Pressure threshold is 5000 psi — exceeding this requires \
operator intervention
- Alert band is 4800-5000 psi — pressure in this range warrants attention
- net_subsurface (injection minus production) is the reservoir fill rate — \
when positive and sustained, pressure rises toward threshold
- cumulative_net reflects total reservoir pressurisation history — \
high values indicate elevated baseline pressure
- dP_dt_1h and dP_dt_6h indicate current pressure trajectory — \
positive and sustained means threshold approach is likely
- headroom is distance to 5000 psi — values below 200 indicate \
imminent threshold approach
- If headroom is already negative, threshold has been exceeded

Classification rules:
- "above" means pressure is predicted to exceed 5000 psi at that horizon
- "below" means pressure is predicted to remain below 5000 psi
- confidence is your certainty in the classification [0, 1]
- Longer horizons should generally have lower confidence
- If current headroom < 200 and dP_dt_6h > 0, threshold crossing \
within 6h is likely
- If pressure is already above threshold, near horizons should be "above" \
with high confidence

No markdown, no prose, no code fences, no extra keys.
Required JSON schema:
{
  "t+1h":  {"classification": "above" or "below", "confidence": <0_to_1_float>},
  "t+6h":  {"classification": "above" or "below", "confidence": <0_to_1_float>},
  "t+12h": {"classification": "above" or "below", "confidence": <0_to_1_float>},
  "t+24h": {"classification": "above" or "below", "confidence": <0_to_1_float>}
}
"""

FEAT_COLS = [
    'Injection Pressure',   # current pressure — distance to threshold is key
    'Net Flow',             # current injection rate
    'net_subsurface',       # injection - production → fill rate
    'cumulative_net',       # total reservoir fill → baseline pressure
    'dP_dt_1h',             # current pressure velocity
    'dP_dt_6h',             # medium-term pressure trend
    'headroom',             # distance to 5000 psi — most direct signal
    'in_alert_band',        # binary: currently between 4800-5000 psi
    'P_roll_std_24h',       # is pressure stable or volatile?
]


# =============================================================================
# PREPARE FEATURES
# =============================================================================
def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['net_subsurface'] = out[INJECTION_FLOW] - out[PRODUCER_FLOW]
    out['cumulative_net'] = out['net_subsurface'].cumsum()
    out['dP_dt_1h']       = out[PRESSURE_COL].diff(1).fillna(0)
    out['dP_dt_6h']       = (out[PRESSURE_COL].diff(6) / 6).fillna(0)
    out['headroom']       = PRESSURE_THRESHOLD - out[PRESSURE_COL]
    out['in_alert_band']  = (
        (out[PRESSURE_COL] >= PRESSURE_ALERT) &
        (out[PRESSURE_COL] <  PRESSURE_THRESHOLD)
    ).astype(int)
    return out


def get_ground_truth_label(pressure_val: float) -> str:
    return "above" if pressure_val >= PRESSURE_THRESHOLD else "below"


# =============================================================================
# FEW-SHOT EXAMPLES
# Cover all four operationally meaningful states:
#   1. Pressure rising, below alert     → all horizons "below" with uncertainty
#   2. Pressure in alert band, rising   → near horizons "above", far uncertain
#   3. Pressure above threshold         → all "above" with high confidence
#   4. Pressure falling, above threshold→ near "above", far "below"
# =============================================================================
def build_few_shot_examples(df:         pd.DataFrame,
                             feat_cols:  list[str],
                             horizons:   list[int] = HORIZONS) -> list[dict]:
    max_h   = max(horizons)
    valid   = df.iloc[:-(max_h + 1)]
    examples = []

    scenarios = [
        # (name, filter_condition)
        ("rising_below_alert",
         (valid[PRESSURE_COL] < PRESSURE_ALERT) &
         (valid['dP_dt_6h'] > 0)),

        ("in_alert_band",
         (valid[PRESSURE_COL] >= PRESSURE_ALERT) &
         (valid[PRESSURE_COL] <  PRESSURE_THRESHOLD) &
         (valid['dP_dt_1h'] > 0)),

        ("above_threshold",
         valid[PRESSURE_COL] >= PRESSURE_THRESHOLD),

        ("falling_from_above",
         (valid[PRESSURE_COL] >= PRESSURE_THRESHOLD) &
         (valid['dP_dt_1h'] < 0)),
    ]

    for name, condition in scenarios:
        candidates = valid[condition]
        if len(candidates) == 0:
            continue

        idx = candidates.index[0]
        loc = df.index.get_loc(idx)
        row = df.loc[idx]

        output = {}
        for h in horizons:
            future_pressure = float(df.iloc[loc + h][PRESSURE_COL])
            label           = get_ground_truth_label(future_pressure)

            # Confidence heuristic based on headroom and trajectory
            headroom_now = float(row['headroom'])
            dp_6h        = float(row['dP_dt_6h'])

            if label == "above" and headroom_now < 0:
                # Already above threshold — high confidence at short horizons
                conf = round(max(0.60, 0.95 - h * 0.01), 2)
            elif label == "above" and headroom_now < 200 and dp_6h > 0:
                # In alert band, rising — confident crossing coming
                conf = round(max(0.55, 0.88 - h * 0.015), 2)
            elif label == "below" and headroom_now > 500:
                # Well below threshold — confident it stays below
                conf = round(max(0.55, 0.90 - h * 0.012), 2)
            else:
                # Uncertain zone
                conf = round(max(0.50, 0.75 - h * 0.015), 2)

            output[f't+{h}h'] = {
                'classification': label,
                'confidence'    : conf,
            }

        examples.append({
            't'              : idx.strftime('%-m/%-d/%Y %I:%M:%S %p'),
            'scenario'       : name,
            'input_features' : {
                col: round(float(row[col]), 4)
                for col in feat_cols
                if col in df.columns
            },
            'output_json'    : output,
        })

        if len(examples) >= 3:   # 3 examples covers most regimes
            break

    return examples


# =============================================================================
# PROMPT BUILDER
# =============================================================================
def build_prompt(time_t:       str,
                 feature_names: list[str],
                 features_t:   dict,
                 horizons:     list[int] = HORIZONS,
                 few_shot:     list[dict] | None = None) -> str:

    horizons_str  = ', '.join([f't+{h}h' for h in horizons])
    features_json = json.dumps(feature_names)

    # Compute operational status from features for explicit guidance
    headroom     = features_t.get('headroom', 9999)
    dp_1h        = features_t.get('dP_dt_1h', 0)
    dp_6h        = features_t.get('dP_dt_6h', 0)
    in_alert     = features_t.get('in_alert_band', 0)
    current_p    = features_t.get('Injection Pressure', 0)

    # Derive plain-language operational status to guide LLM
    if current_p >= PRESSURE_THRESHOLD:
        status = f"ABOVE THRESHOLD — pressure is {abs(headroom):.1f} psi over 5000 psi limit"
    elif in_alert:
        status = f"IN ALERT BAND — {headroom:.1f} psi below threshold, pressure {'rising' if dp_1h > 0 else 'falling'}"
    elif headroom < 500 and dp_6h > 0:
        status = f"APPROACHING THRESHOLD — {headroom:.1f} psi headroom, sustained upward trend"
    else:
        status = f"NORMAL — {headroom:.1f} psi headroom, pressure {'rising' if dp_1h > 0 else 'stable/falling'}"

    prompt_lines = [
        f'Task: Classify whether Injection Pressure will exceed '
        f'5000 psi at horizons: {horizons_str}',
        f'Current timestamp: {time_t}',
        f'Operational status: {status}',
        '',
        'Key signals:',
        f'  Current pressure : {current_p:.2f} psi  '
        f'(threshold = {PRESSURE_THRESHOLD:.0f} psi)',
        f'  Headroom         : {headroom:.2f} psi',
        f'  dP/dt last 1h    : {dp_1h:+.2f} psi/hr  '
        f'({"rising" if dp_1h > 0 else "falling or stable"})',
        f'  dP/dt last 6h    : {dp_6h:+.2f} psi/hr  '
        f'({"sustained rise" if dp_6h > 0 else "sustained fall or stable"})',
        '',
        f'Full feature snapshot at time t '
        f'(use only these — no external knowledge):',
        json.dumps(features_t, indent=2),
        '',
        'Classify "above" if pressure will exceed 5000 psi, '
        '"below" if it will not.',
        'Confidence should reflect trajectory clarity — '
        'high when trend is strong and sustained, '
        'lower when pressure is near threshold or trend is reversing.',
        '',
        'Important: Return JSON only. No explanation text.',
        'Required output schema:',
        json.dumps(
            {f't+{h}h': {
                'classification': 'above or below',
                'confidence'    : '0_to_1_float'}
             for h in horizons},
            indent=2
        ),
    ]

    if few_shot:
        prompt_lines += ['', '=' * 40, 'Examples from historical data:', '=' * 40]
        for i, ex in enumerate(few_shot, start=1):
            prompt_lines += [
                '',
                f'Example {i} ({ex.get("scenario", "")}) '
                f'— input at t={ex["t"]}:',
                json.dumps(ex['input_features'], indent=2),
                f'Example {i} — correct output:',
                json.dumps(ex['output_json'], indent=2),
            ]
        prompt_lines.append('=' * 40)

    prompt_lines += ['', 'Now classify the following input:', '']

    return '\n'.join(prompt_lines)


# =============================================================================
# PARSE AND EVALUATE RESPONSE
# =============================================================================
def parse_response(response_text: str,
                   horizons: list[int] = HORIZONS) -> dict | None:
    """
    Parse LLM classification response.
    Returns dict keyed by horizon with classification and confidence,
    or None if parsing fails.
    """
    try:
        text = response_text.strip()
        # Strip accidental markdown fences
        if '```' in text:
            text = text.split('```')[1]
            if text.startswith('json'):
                text = text[4:]
        text = text.strip()

        parsed = json.loads(text)

        result = {}
        for h in horizons:
            key = f't+{h}h'
            if key not in parsed:
                print(f"  ⚠ Missing key {key}")
                return None
            val  = parsed[key]
            clf  = val.get('classification', '').lower().strip()
            conf = float(val.get('confidence', 0.5))

            if clf not in ('above', 'below'):
                print(f"  ⚠ Invalid classification '{clf}' for {key}")
                return None

            result[key] = {
                'classification': clf,
                'confidence'    : round(conf, 4),
                'binary'        : 1 if clf == 'above' else 0,
            }
        return result

    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"  ⚠ Parse error: {e}")
        print(f"  Raw response: {response_text[:200]}")
        return None


def evaluate_classifications(results_df:  pd.DataFrame,
                              horizons:    list[int] = HORIZONS,
                              threshold:   float = PRESSURE_THRESHOLD) -> pd.DataFrame:
    """
    Compute TP/FP/FN/TN + Precision/Recall/F1 per horizon.
    Mirrors the KF confusion matrix evaluation exactly
    so results are directly comparable.
    """
    rows = []
    for h in horizons:
        pred_col  = f't+{h}h_binary'
        # Ground truth: was actual pressure >= threshold at t+h?
        # Requires pressure column shifted back h steps
        actual_col = f'actual_above_{h}h'

        if pred_col not in results_df.columns:
            continue
        if actual_col not in results_df.columns:
            continue

        valid = results_df[[pred_col, actual_col]].dropna()
        if len(valid) == 0:
            continue

        y_pred = valid[pred_col].astype(int)
        y_true = valid[actual_col].astype(int)

        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
        recall    = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        f1        = (2 * precision * recall / (precision + recall)
                     if not (np.isnan(precision) or np.isnan(recall))
                     and (precision + recall) > 0 else float('nan'))

        rows.append({
            'Horizon'  : f'+{h}h',
            'TP'       : tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'Precision': round(precision, 3) if not np.isnan(precision) else '—',
            'Recall'   : round(recall, 3)    if not np.isnan(recall)    else '—',
            'F1'       : round(f1, 3)        if not np.isnan(f1)        else '—',
        })

    return pd.DataFrame(rows)
    

In [9]:
from openai import OpenAI
import json

API_KEY = 'EMPTY'

MODEL_ENDPOINTS = [
    {
        'label'      : 'gemma-3-12b-it',
        'model_name' : 'gemma-3-12b-it',
        'host'       : 'localhost',
        'port'       : 8002
    },
]

endpoint_cfg = MODEL_ENDPOINTS[0]

def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = OpenAI(
        api_key  = API_KEY,
        base_url = f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model       = endpoint_cfg['model_name'],
        messages    = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ],
        temperature = temperature,
        seed        = seed,
    )
    return response.choices[0].message.content

In [10]:
# =============================================================================
# CONFIGURATION
# =============================================================================
WEEK3_TEST_CSV = 'week3_artifacts/FTES_1hour_cleaned.csv'

TIME_COL       = 'Time'
PRESSURE_COL   = 'Injection Pressure'
INJECTION_FLOW = 'Net Flow'
PRODUCER_FLOW  = 'TN Interval Flow'

PRESSURE_THRESHOLD = 5000.0
PRESSURE_ALERT     = 4800.0
HORIZONS           = [1, 6, 12, 24]

# =============================================================================
# LOAD & PREPARE — minimal, only what LLM needs
# =============================================================================
def load_and_prepare(csv_path: str) -> pd.DataFrame:
    """
    Load raw well data and engineer the features the LLM needs.
    Output contains only TIME_COL + FEAT_COLS + ground truth labels.
    Nothing else.
    """
    # Load only the three raw columns needed
    raw = pd.read_csv(csv_path, parse_dates=[TIME_COL])
    raw = raw.sort_values(TIME_COL).reset_index(drop=True)

    missing = {PRESSURE_COL, INJECTION_FLOW, PRODUCER_FLOW} - set(raw.columns)
    if missing:
        raise ValueError(f'Missing raw columns: {sorted(missing)}')

    df = raw[[TIME_COL, PRESSURE_COL, INJECTION_FLOW, PRODUCER_FLOW]].copy()

    # Engineer features
    df['net_subsurface'] = df[INJECTION_FLOW] - df[PRODUCER_FLOW]
    df['cumulative_net'] = df['net_subsurface'].cumsum()
    df['dP_dt_1h']       = df[PRESSURE_COL].diff(1).fillna(0)
    df['dP_dt_6h']       = (df[PRESSURE_COL].diff(6) / 6).fillna(0)
    df['headroom']       = PRESSURE_THRESHOLD - df[PRESSURE_COL]
    df['in_alert_band']  = (
        (df[PRESSURE_COL] >= PRESSURE_ALERT) &
        (df[PRESSURE_COL] <  PRESSURE_THRESHOLD)
    ).astype(int)
    df['P_roll_std_24h'] = df[PRESSURE_COL].rolling(24).std().fillna(0)

    # Ground truth labels for evaluation
    for h in HORIZONS:
        df[f'actual_above_{h}h'] = (
            df[PRESSURE_COL]
            .shift(-h)
            .apply(lambda p: 1 if not pd.isna(p) and p >= PRESSURE_THRESHOLD else 0)
        )

    # Drop NaN rows from rolling window warmup and horizon tail
    feat_cols   = ['net_subsurface', 'cumulative_net',
                   'dP_dt_1h', 'dP_dt_6h',
                   'headroom', 'in_alert_band']
    label_cols  = [f'actual_above_{h}h' for h in HORIZONS]
    df = df.dropna(subset=feat_cols + label_cols).reset_index(drop=True)

    print(f'Loaded           : {csv_path}')
    print(f'Rows for eval    : {len(df)}')
    print(f'Period           : {df[TIME_COL].iloc[0]} → {df[TIME_COL].iloc[-1]}')
    print(f'\nClass balance:')
    for h in HORIZONS:
        n_above = int(df[f'actual_above_{h}h'].sum())
        print(f'  t+{h}h : {n_above} above / {len(df)-n_above} below '
              f'({100*n_above/len(df):.1f}% above threshold)')
    return df


# These are the only columns passed to the LLM
FEAT_COLS = [
    'Injection Pressure',
    'Net Flow',
    'net_subsurface',
    'cumulative_net',
    'dP_dt_1h',
    'dP_dt_6h',
    'headroom',
    'in_alert_band',
]

# Match their function signature exactly
def call_llm(prompt: str, endpoint_cfg: dict) -> str:
    return query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0)

# Their build_prompt uses 'version' parameter — add that to yours
# or just hardcode since you only have one version

selected_prompt_ver = 'v1'  # match their version variable name
endpoint_cfg = MODEL_ENDPOINTS[0]  # gemma-3-12b-it

# =============================================================================
# RUN
# =============================================================================
df       = load_and_prepare(WEEK3_TEST_CSV)
few_shot = build_few_shot_examples(df.set_index(TIME_COL), FEAT_COLS, HORIZONS)


# ── single row test ───────────────────────────────────────────────────────────
example = df.iloc[100]  # pick a row where pressure is near threshold
                         # more interesting than row 0 which may be early data

time_t     = example[TIME_COL].strftime('%-m/%-d/%Y %I:%M:%S %p')
features_t = {col: round(float(example[col]), 4) for col in FEAT_COLS}

prompt = build_prompt(
    time_t        = time_t,
    feature_names = FEAT_COLS,
    features_t    = features_t,
    horizons      = HORIZONS,
    few_shot      = few_shot,
)

print("=" * 60)
print("PROMPT:")
print("=" * 60)
print(prompt)
print("=" * 60)

raw    = query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0)
parsed = parse_response(raw, HORIZONS)

print(f"\nRaw response:")
print(raw)
print(f"\nParsed:")
print(parsed)
print(f"\nGround truth:")
for h in HORIZONS:
    print(f"  t+{h}h actual_above: {int(example[f'actual_above_{h}h'])}")

Loaded           : week3_artifacts/FTES_1hour_cleaned.csv
Rows for eval    : 1702
Period           : 2024-12-13 20:00:00 → 2025-02-22 17:00:00

Class balance:
  t+1h : 520 above / 1182 below (30.6% above threshold)
  t+6h : 520 above / 1182 below (30.6% above threshold)
  t+12h : 520 above / 1182 below (30.6% above threshold)
  t+24h : 520 above / 1182 below (30.6% above threshold)
PROMPT:
Task: Classify whether Injection Pressure will exceed 5000 psi at horizons: t+1h, t+6h, t+12h, t+24h
Current timestamp: 12/18/2024 12:00:00 AM
Operational status: NORMAL — 714.6 psi headroom, pressure rising

Key signals:
  Current pressure : 4285.37 psi  (threshold = 5000 psi)
  Headroom         : 714.63 psi
  dP/dt last 1h    : +9.03 psi/hr  (rising)
  dP/dt last 6h    : +2.32 psi/hr  (sustained rise)

Full feature snapshot at time t (use only these — no external knowledge):
{
  "Injection Pressure": 4285.374,
  "Net Flow": 2.5025,
  "net_subsurface": 1.6686,
  "cumulative_net": 144.4505,
  "dP_dt_

In [11]:
results  = []
n_failed = 0

for i, row in df.iterrows():
    time_t     = row[TIME_COL].strftime('%-m/%-d/%Y %I:%M:%S %p')
    features_t = {col: round(float(row[col]), 4) for col in FEAT_COLS}

    prompt = build_prompt(
        time_t        = time_t,
        feature_names = FEAT_COLS,
        features_t    = features_t,
        horizons      = HORIZONS,
        few_shot      = few_shot,
    )

    raw    = query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw, HORIZONS)

    if parsed is None:
        n_failed += 1
        continue

    result_row = {TIME_COL: row[TIME_COL]}
    for h in HORIZONS:
        key = f't+{h}h'
        result_row[f'{key}_classification'] = parsed[key]['classification']
        result_row[f'{key}_confidence']     = parsed[key]['confidence']
        result_row[f'{key}_binary']         = parsed[key]['binary']
        result_row[f'actual_above_{h}h']    = row[f'actual_above_{h}h']
    results.append(result_row)

    if i % 100 == 0:
        print(f'  {i}/{len(df)} rows  ({n_failed} failed)')

results_df = pd.DataFrame(results)
print(f'\nComplete: {len(results_df)} rows, {n_failed} failed parses')

# Save
results_df.to_csv(f'llm_pressure_classifications_{ITERATION_NUMBER}.csv',
    index=False)

# Evaluate
eval_metrics = evaluate_classifications(results_df, HORIZONS)
print('\n' + '=' * 55)
print('  LLM THRESHOLD CLASSIFICATION METRICS')
print('  Compare to KF:')
print('  KF Validate +1h  : P=0.990 R=1.000 F1=0.995')
print('  KF Validate +6h  : P=0.959 R=1.000 F1=0.979')
print('  KF Validate +12h : P=0.938 R=1.000 F1=0.968')
print('  KF Validate +24h : P=0.872 R=0.851 F1=0.861')
print('=' * 55)
print(eval_metrics.to_string(index=False))

  0/1702 rows  (0 failed)
  100/1702 rows  (0 failed)
  200/1702 rows  (0 failed)
  300/1702 rows  (0 failed)
  400/1702 rows  (0 failed)
  500/1702 rows  (0 failed)
  600/1702 rows  (0 failed)
  700/1702 rows  (0 failed)
  800/1702 rows  (0 failed)
  900/1702 rows  (0 failed)
  1000/1702 rows  (0 failed)
  1100/1702 rows  (0 failed)
  1200/1702 rows  (0 failed)
  1300/1702 rows  (0 failed)
  1400/1702 rows  (0 failed)
  1500/1702 rows  (0 failed)
  1600/1702 rows  (0 failed)
  1700/1702 rows  (0 failed)

Complete: 1702 rows, 0 failed parses


NameError: name 'OUTPUT_DIR' is not defined

In [13]:
# Save
results_df.to_csv(f'llm_pressure_classifications.csv',
    index=False)

### Parse response

In [9]:
def parse_response(raw_text):
    output = {'t+1': None, 'llm_prediction': np.nan, 'llm_confidence': np.nan, 'parse_ok': False, 'parse_error': None}

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output

    m = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        output['parse_error'] = 'invalid_json'
        return output

    t_plus_1 = payload.get('t+1', None)
#     try:
#         t_plus_1 = datetime.strptime(t_plus_1, "%m/%d/%Y %I:%M:%S %p")
#     except Exception:
#         t_plus_1 = None

    pred = payload.get('prediction', np.nan)
    try:
        pred = float(pred)
    except Exception:
        pred = np.nan

    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except Exception:
        conf = np.nan

    if not np.isnan(conf) and not (0 <= conf <= 1):
        conf = np.nan
    
    output['t+1'] = t_plus_1
    output['llm_prediction'] = pred
    output['llm_confidence'] = conf
    output['parse_ok'] = True
    return output

## Step 5: Single-row smoke test

What this does: tests one example on each model endpoint before running the full benchmark loop.

Why it matters: this catches endpoint/port issues early and confirms all models are reachable.

Principle: validate connectivity and output format for each model before large-scale evaluation.

In [ ]:
# Configure prompt version to run/save
selected_prompt_ver = 'v1'

In [10]:
def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed
    )
    return response.choices[0].message.content

example = df.iloc[0]
for endpoint_cfg in MODEL_ENDPOINTS:
    raw = query_llm(build_prompt(example['Time'], 
                                 FEAT_COLS, 
                                 example[FEAT_COLS].to_dict(), 
                                 version=selected_prompt_ver,
                                 few_shot=few_shot_samples), 
                    endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw)
    print(f"\nModel: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print('Raw response:')
    print(raw)
    print('Parsed:')
    print(parsed)


Model: gemma-3-12b-it (port 8002)
Raw response:
```json
{
"t+1": "2/12/2025 12:00:00 AM",
"prediction": 4982.4312,
"confidence": 0.88
}
```
Parsed:
{'t+1': '2/12/2025 12:00:00 AM', 'llm_prediction': 4982.4312, 'llm_confidence': 0.88, 'parse_ok': True, 'parse_error': None}


### Save prompts

In [11]:
prompt_example = build_prompt("{time_t}", FEAT_COLS, {feat: "{float}" for feat in FEAT_COLS}, version=selected_prompt_ver, few_shot=few_shot_samples)
prompt_template_path = PROMPT_DIR / f'prompt_template_{selected_prompt_ver}.txt'
prompt_template_path.write_text(prompt_example, encoding='utf-8')


few_shot_path = PROMPT_DIR / f'few_shot_samples_{selected_prompt_ver}.json'
few_shot_path.write_text(json.dumps(few_shot_samples, indent=2), encoding='utf-8')

1581

## Step 6: Full benchmark run

What this does: loops over all configured model endpoints and all test rows, storing raw responses and parsed predictions.

Why it matters: this produces side-by-side model outputs from a single consistent pipeline.

Principle: hold the dataset and prompt constant while varying models for a fair model-to-model comparison.

In [35]:
rows = []
for endpoint_cfg in MODEL_ENDPOINTS:
    print(f"\nRunning benchmark for {endpoint_cfg['model_name']} on port {endpoint_cfg['port']}...")
    for i, row in df.iterrows():
        raw = query_llm(
            build_prompt(example['Time'], FEAT_COLS, example[FEAT_COLS].to_dict(), version=selected_prompt_ver, few_shot=few_shot_samples),
            endpoint_cfg,
            temperature=0.0,
            seed=0
        )
        parsed = parse_response(raw)
        
        if isinstance(row[TRUE_VAL_COL], float):
            ground_truth = row[TRUE_VAL_COL]
        else:
            ground_truth = row[TRUE_VAL_COL].iloc[0]

        rows.append({
            TIME_COL: row[TIME_COL],
            'model_label': endpoint_cfg['label'],
            'model_name': endpoint_cfg['model_name'],
            'endpoint_port': endpoint_cfg['port'],
            'prompt_version': selected_prompt_ver,
            'raw_response': raw,
            **parsed,
            'ground_truth': ground_truth,
            'ml_prediction': row[ML_PRED_COL]
        })

        if (i + 1) % 10 == 0:
            print(f"  Completed {i + 1}/{len(df)}")
            
    print(f"{endpoint_cfg['model_name']} benchmark run complete.")


Running benchmark for gemma-3-12b-it on port 8002...
  Completed 10/256
  Completed 20/256
  Completed 30/256
  Completed 40/256
  Completed 50/256
  Completed 60/256
  Completed 70/256
  Completed 80/256
  Completed 90/256
  Completed 100/256
  Completed 110/256
  Completed 120/256
  Completed 130/256
  Completed 140/256
  Completed 150/256
  Completed 160/256
  Completed 170/256
  Completed 180/256
  Completed 190/256
  Completed 200/256
  Completed 210/256
  Completed 220/256
  Completed 230/256
  Completed 240/256
  Completed 250/256
gemma-3-12b-it benchmark run complete.


In [44]:
results_df = pd.DataFrame(rows)

print("Original results")   
display(results_df.head())
display(results_df.tail())

results_df.to_csv(RAW_RESULTS_PATH, index=False)
print(f'Saved raw CSV: {RAW_RESULTS_PATH}')

Original results


,Time,model_label,model_name,endpoint_port,prompt_version,raw_response,t+1,llm_prediction,llm_confidence,parse_ok,parse_error,ground_truth,ml_prediction
0,2025-02-12 02:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4981.276052,NaN
1,2025-02-12 03:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4980.895880,4981.380412
2,2025-02-12 04:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4981.522123,4980.847361
3,2025-02-12 05:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4981.172066,4981.444753
4,2025-02-12 06:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4980.811139,4981.104882


,Time,model_label,model_name,endpoint_port,prompt_version,raw_response,t+1,llm_prediction,llm_confidence,parse_ok,parse_error,ground_truth,ml_prediction
251,2025-02-22 13:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4950.528957,4951.339960
252,2025-02-22 14:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4952.101775,4950.599438
253,2025-02-22 15:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4952.104437,4952.357578
254,2025-02-22 16:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4951.670969,4952.028750
255,2025-02-22 17:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4951.120372,NaN


Saved raw CSV: /home/jupyter-theresa.pham/ESSD_AI_Competition_1/SubTerraTensorTinkerers/Results/gemma3_results_raw_1.csv


In [45]:
try: 
    clean_df = results_df.copy()
except:
    results_df = pd.read_csv(RAW_RESULTS_PATH)
    clean_df = results_df.copy()
    print(f"Loaded raw results from {CLEAN_RESULTS_PATH}")

clean_df[['llm_prediction', 'llm_confidence', 'parse_ok', 'parse_error']] = clean_df[['llm_prediction', 'llm_confidence', 'parse_ok', 'parse_error']].shift(1)
clean_df = clean_df.iloc[1:-1]
clean_df = clean_df.drop(columns=["raw_response", "t+1", "parse_ok", "parse_error"])

print("Clean results")
display(clean_df.head())

clean_df.to_csv(CLEAN_RESULTS_PATH, index=False)
print(f'Saved clean CSV: {CLEAN_RESULTS_PATH}')

Clean results


,Time,model_label,model_name,endpoint_port,prompt_version,llm_prediction,llm_confidence,ground_truth,ml_prediction
1,2025-02-12 03:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4980.895880,4981.380412
2,2025-02-12 04:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4981.522123,4980.847361
3,2025-02-12 05:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4981.172066,4981.444753
4,2025-02-12 06:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4980.811139,4981.104882
5,2025-02-12 07:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4979.714603,4980.695511


Saved clean CSV: /home/jupyter-theresa.pham/ESSD_AI_Competition_1/SubTerraTensorTinkerers/Results/gemma3_results_clean_1.csv


## Step 7: Evaluate and compare

What this does: computes ML error, LLM error, and invalid output rate from the parsed results.

Why it matters: this gives a direct Week 3 vs Week 4 comparison on the same data split.

Principle: evaluate both quality and reliability. Accuracy without format reliability can still fail in production workflows.

In [46]:
try: 
    eval_df = clean_df.copy()
except:
    clean_df = pd.read_csv(CLEAN_RESULTS_PATH)
    eval_df = clean_df.copy()
    print(f"Loaded cleaned results from {CLEAN_RESULTS_PATH}")


def reg_metrics(y_true, y_pred):
    return {
        "rmse": np.sqrt(np.mean((y_true - y_pred)**2)),
        "mae": np.mean(np.abs(y_true - y_pred)),
    }

summary_rows = []
group_cols = ["model_label", "model_name", "endpoint_port"]

for keys, g in eval_df.groupby(group_cols, dropna=False):
    valid_llm = g.dropna(subset=["ground_truth", "llm_prediction"])

    # Compare ML and LLM on the exact same valid rows for fairness
    valid_ml = valid_llm.dropna(subset=["ml_prediction"])
    if len(valid_ml) > 0:
        ml_m = reg_metrics(valid_ml["ground_truth"], valid_ml["ml_prediction"])
    else:
        ml_m = {"rmse": np.nan, "mae": np.nan}

    summary_rows.append({
        "model_type": "Week3_ML",
        "model_label": "week3-xgb",
        "model_name": "Week3 ML - XGB",
        "endpoint_port": np.nan,
        "rows_total": int(len(g)),
        "rows_valid": int(len(valid_ml)),
        "invalid_rate": float(1 - len(valid_ml) / len(g)),
        "rmse": ml_m["rmse"],
        "mae": ml_m["mae"]
    })
    
    llm_m = reg_metrics(valid_llm["ground_truth"], valid_llm["llm_prediction"])
    
    summary_rows.append({
        "model_type": "Week4_LLM",
        "model_label": keys[0],
        "model_name": keys[1],
        "endpoint_port": keys[2],
        "rows_total": int(len(g)),
        "rows_valid": int(len(valid_llm)),
        "invalid_rate": float(1 - len(valid_llm) / len(g)),
        "rmse": llm_m["rmse"],
        "mae": llm_m["mae"],
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
                                
summary_df.to_csv(SUMMARY_RESULTS_PATH, index=False)
print(f'Saved summary CSV: {SUMMARY_RESULTS_PATH}')

,model_type,model_label,model_name,endpoint_port,rows_total,rows_valid,invalid_rate,rmse,mae
0,Week3_ML,week3-xgb,Week3 ML - XGB,NaN,254,254,0.0,1.027400,0.740474
1,Week4_LLM,gemma-3-12b-it,gemma-3-12b-it,8002.0,254,254,0.0,18.505921,17.028993


Saved summary CSV: /home/jupyter-theresa.pham/ESSD_AI_Competition_1/SubTerraTensorTinkerers/Results/gemma3_results_summary_1.csv
